# 🔍 FAQ Semantic Search Engine

A **semantic search engine** over the HuggingFace `wiki_qa` dataset using sentence embeddings.  
Finds answers even when your query uses different words than the FAQ.

---
**Flow:**  
`Query` → `Embed with all-MiniLM-L6-v2` → `Cosine Similarity` → `Top-N Results`

> Run cells **top to bottom** on first use. Embeddings are cached — subsequent runs skip encoding.

## Cell 1 — Install Dependencies

In [1]:
import subprocess, sys

packages = [
    "sentence-transformers>=2.7.0",
    "datasets>=2.19.0",
    "numpy>=1.26.0",
    "tf-keras",           # compatibility shim for Keras 3 + transformers
]

for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

print("All dependencies ready.")

All dependencies ready.


## Cell 2 — Imports & Config

In [2]:
import os
import json
import warnings
import numpy as np
from pathlib import Path

# Suppress noisy TF / oneDNN warnings
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")
warnings.filterwarnings("ignore")

# ── Configuration ───────────────────────────────────────────
MODEL_NAME   = "sentence-transformers/all-MiniLM-L6-v2"   # fast, 80 MB, runs locally
DATASET_NAME = "wiki_qa"                                   # public HuggingFace dataset
TOP_K        = 5                                           # default number of results
CACHE_DIR    = Path("./cache")
EMB_FILE     = CACHE_DIR / "embeddings.npy"
META_FILE    = CACHE_DIR / "metadata.json"

print("Config loaded.")

Config loaded.


## Cell 3 — Load Dataset from HuggingFace

In [ ]:
from datasets import load_dataset

print(f"Loading '{DATASET_NAME}' from HuggingFace...")
ds = load_dataset(DATASET_NAME, split="train")

# Deduplicate by question, keep only rows with non-empty answers
seen = set()
faqs = []
for row in ds:
    q = row["question"].strip()
    a = row["answer"].strip()
    if not q or not a:
        continue
    key = q.lower()
    if key not in seen:
        seen.add(key)
        faqs.append({"question": q, "answer": a})

print(f"Loaded {len(faqs)} unique FAQ entries.")
print("\nSample entry:")
print(f"  Q: {faqs[0]['question']}")
print(f"  A: {faqs[0]['answer'][:120]}...")

## Cell 4 — Load Embedding Model

In [ ]:
from sentence_transformers import SentenceTransformer

print(f"Loading model: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME)
print("Model loaded.")

## Cell 5 — Build / Load Embedding Index

> **First run:** encodes all FAQs and saves to `./cache/` (~30 sec).  
> **Subsequent runs:** loads from cache instantly.

In [ ]:
if EMB_FILE.exists() and META_FILE.exists():
    # ── Load from cache ──────────────────────────────────────
    print("Cache found — loading embeddings from disk...")
    embeddings = np.load(EMB_FILE)
    with open(META_FILE, "r", encoding="utf-8") as f:
        faqs = json.load(f)
    print(f"Loaded {len(faqs)} FAQ embeddings from cache.")

else:
    # ── Build from scratch ───────────────────────────────────
    print(f"Building embeddings for {len(faqs)} FAQs (one-time, please wait)...")
    CACHE_DIR.mkdir(exist_ok=True)

    questions  = [f["question"] for f in faqs]
    embeddings = model.encode(
        questions,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,   # pre-normalize: cosine sim = dot product
    )

    np.save(EMB_FILE, embeddings)
    with open(META_FILE, "w", encoding="utf-8") as f:
        json.dump(faqs, f, ensure_ascii=False, indent=2)

    print(f"[OK] Embeddings saved to {CACHE_DIR}/")

print(f"\nIndex shape: {embeddings.shape}  (FAQs x embedding_dim)")

## Cell 6 — Search Function

In [ ]:
def search(query: str, top_k: int = TOP_K) -> list[dict]:
    """
    Semantic search over the FAQ index.
    Returns a list of dicts: [{rank, score, question, answer}, ...]
    """
    if not query.strip():
        return []

    # Encode query (already normalized)
    query_vec = model.encode([query], normalize_embeddings=True)[0]

    # Cosine similarity via dot product (embeddings pre-normalized)
    scores = embeddings @ query_vec                          # shape: (N,)

    # Get top_k indices, sorted descending
    top_idx = np.argpartition(scores, -top_k)[-top_k:]
    top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]

    return [
        {
            "rank":     int(rank),
            "score":    round(float(scores[i]), 4),
            "question": faqs[i]["question"],
            "answer":   faqs[i]["answer"],
        }
        for rank, i in enumerate(top_idx, start=1)
    ]


def display_results(query: str, results: list[dict]):
    """Pretty-print results in the notebook."""
    from IPython.display import display, HTML

    if not results:
        print("No results found.")
        return

    rows = ""
    for r in results:
        score = r["score"]
        color = (
            "#2ecc71" if score > 0.70
            else "#f39c12" if score > 0.50
            else "#e74c3c"
        )
        answer_preview = r["answer"][:300] + ("..." if len(r["answer"]) > 300 else "")
        rows += f"""
        <tr>
          <td style='text-align:center;font-weight:bold;color:#888'>#{r['rank']}</td>
          <td style='text-align:center;font-weight:bold;color:{color}'>{score:.3f}</td>
          <td style='font-weight:600'>{r['question']}</td>
          <td style='color:#ccc;font-size:0.9em'>{answer_preview}</td>
        </tr>"""

    html = f"""
    <style>
      .faq-table {{ width:100%; border-collapse:collapse; font-family:sans-serif; font-size:14px; }}
      .faq-table th {{ background:#1e1e2e; color:#cdd6f4; padding:10px 14px; text-align:left; }}
      .faq-table td {{ padding:10px 14px; border-bottom:1px solid #313244; vertical-align:top; }}
      .faq-table tr:hover td {{ background:#181825; }}
      .faq-header {{ font-family:sans-serif; color:#cba6f7; margin-bottom:8px; }}
    </style>
    <p class='faq-header'><b>Results for:</b> <i>"{query}"</i> &nbsp;|&nbsp; top {len(results)} matches</p>
    <table class='faq-table'>
      <thead><tr>
        <th>#</th><th>Score</th><th>Matched Question</th><th>Answer</th>
      </tr></thead>
      <tbody>{rows}</tbody>
    </table>"""

    display(HTML(html))


print("Search function ready. Proceed to the query cell below.")

## Cell 7 — Run a Query

**Edit `QUERY` and `TOP_K` below, then run this cell to search.**

In [ ]:
# ── Edit these ──────────────────────────────────────
QUERY = "What is the capital of France?"
TOP_K = 5
# ────────────────────────────────────────────────────

results = search(QUERY, top_k=TOP_K)
display_results(QUERY, results)

## Cell 8 — Interactive Search Widget

Run this cell for a live text-box interface inside the notebook.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    txt   = widgets.Text(placeholder="Type your question here...",
                         description="Query:", layout=widgets.Layout(width="60%"))
    k_sl  = widgets.IntSlider(value=5, min=1, max=20, step=1,
                              description="Top-K:", continuous_update=False)
    btn   = widgets.Button(description="Search", button_style="primary")
    out   = widgets.Output()

    def on_search(_):
        with out:
            clear_output(wait=True)
            q = txt.value.strip()
            if q:
                display_results(q, search(q, top_k=k_sl.value))
            else:
                print("Please enter a query first.")

    btn.on_click(on_search)
    txt.on_submit(on_search)

    display(widgets.VBox([widgets.HBox([txt, k_sl, btn]), out]))

except ImportError:
    print("ipywidgets not installed. Run: pip install ipywidgets")
    print("Falling back — use Cell 7 to enter queries manually.")

---
## Cell 9 — Inspect a Full Answer

Set `RESULT_INDEX` (1-based rank) to read the complete answer for any result.

In [ ]:
RESULT_INDEX = 1   # rank 1 = best match

# Re-uses results from Cell 7
r = results[RESULT_INDEX - 1]
print(f"Rank #{r['rank']}  |  Score: {r['score']:.4f}")
print(f"\nQ: {r['question']}")
print(f"\nA:\n{r['answer']}")